# 1. Importação das bibliotecas e carregamento dos dados

### 1. Biblioteca

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

###2. Carregando os dados

In [3]:
df_analise = pd.read_csv('https://github.com/JCDIAMANTINO/Datathon-Passos-Magicos-Fase-5/raw/refs/heads/main/Dados_datathon/pede_completo.csv', sep=';', decimal=',')

df_analise.head()

,ano,ra,nome,idade,genero,fase,turma,ano_ingresso,instituicao_ensino,inde,...,fase_ideal,defasagem,iaa,ieg,ips,ida,ipv,ian,ipp,risco_defasagem
0,2022,RA-1,Aluno-1,19,Feminino,7,A,2016,Pública,5.783,...,8,-1.0,8.3,4.1,5.6,4.0,7.278,5.0,7.708333,1
1,2022,RA-2,Aluno-2,17,Feminino,7,A,2017,Privada,7.055,...,7,0.0,8.8,5.2,6.3,6.8,6.778,10.0,7.708333,0
2,2022,RA-3,Aluno-3,17,Feminino,7,A,2016,Privada,6.591,...,7,0.0,0.0,7.9,5.6,5.6,7.556,10.0,7.708333,1
3,2022,RA-4,Aluno-4,17,Masculino,7,A,2017,Privada,5.951,...,7,0.0,8.8,4.5,5.6,5.0,5.278,10.0,7.708333,1
4,2022,RA-5,Aluno-5,17,Feminino,7,A,2016,Privada,7.427,...,7,0.0,7.9,8.6,5.6,5.2,7.389,10.0,7.708333,0


In [4]:
# Criando categorização de Defasagem Conforme o Relatório PEDE: >= 0 (Em dia), -1 a -2 (Moderada), <= -3 (Severa)
def categorizar_defasagem(d):
    if d >= 0: return 'Em fase / Adequado'
    elif d >= -2: return 'Defasagem Moderada'
    else: return 'Defasagem Severa'

df_analise['status_ian'] = df_analise['defasagem'].apply(categorizar_defasagem)

# 2. Bloco de Perguntas

### Pergunta 1: Qual é o perfil geral de defasagem (IAN) e como evolui ao longo do ano?

In [5]:
# Função de categorização
def categorizar_ian_oficial(nota_ian):
    if nota_ian >= 9.0:
        return 'Adequado (Nota 10)'
    elif nota_ian >= 4.0:
        return 'Defasagem Moderada (Nota 5)'
    else:
        return 'Defasagem Severa (Nota 2.5 ou 0)'

df_analise['status_ian'] = df_analise['ian'].apply(categorizar_ian_oficial)

# Cálculos: Agrupando e calculando o percentual
df_q1 = df_analise.groupby(['ano', 'status_ian']).size().reset_index(name='contagem')

# Calcula o total de alunos por ano
totais_por_ano = df_q1.groupby('ano')['contagem'].transform('sum')

# Cria a coluna de percentual e o texto do rótulo
df_q1['percentual'] = (df_q1['contagem'] / totais_por_ano) * 100
df_q1['rotulo'] = df_q1.apply(lambda x: f"{int(x['contagem'])} ({x['percentual']:.1f}%)", axis=1)

# Gráfico de Barras Agrupadas com os novos rótulos
fig1 = px.bar(df_q1, x='ano', y='contagem', color='status_ian',
              title="1. Adequação de Nível (IAN): Perfil de Defasagem ao Longo dos Anos",
              barmode='group',
              text='rotulo',
              color_discrete_map={
                  'Adequado (Nota 10)': 'green',
                  'Defasagem Moderada (Nota 5)': 'orange',
                  'Defasagem Severa (Nota 2.5 ou 0)': 'red'
              },
              category_orders={'status_ian': ['Adequado (Nota 10)', 'Defasagem Moderada (Nota 5)', 'Defasagem Severa (Nota 2.5 ou 0)']})

fig1.update_traces(textposition='outside')

fig1.update_layout(yaxis=dict(title='Quantidade de Alunos'), margin=dict(t=50))

fig1.show()

**Insight Analítico:** O gráfico evidencia a evolução do programa ao longo dos anos. Em 2022 havia um número significativo de alunos em defasagem moderada e severa. Em 2023 e 2024 houve grande migração de alunos para o nível 'Adequado', demonstrando a eficácia da intervenção pedagógica da Passos Mágiocos.

### Pergunta 2: O desempenho acadêmico médio (IDA) está melhorando, estagnado ou caindo ao longo das fases e anos?

In [7]:
# Calculando a média e preparando os dados
df_q2 = df_analise.groupby(['ano', 'fase'])['ida'].mean().reset_index()
df_q2['ida_medio'] = df_q2['ida'].round(2)
df_q2 = df_q2.sort_values(by='fase')
df_q2['Nome da Fase'] = df_q2['fase'].apply(lambda x: 'Alfa' if x == 0 else f'Fase {x}')
df_q2['ano'] = df_q2['ano'].astype(str)

# Descobrindo o valor mínimo e máximo para aplicar o Zoom no Eixo Y
min_y = df_q2['ida_medio'].min() - 0.5
max_y = df_q2['ida_medio'].max() + 0.5

# Criando o Gráfico (Ordem e Cores Ajustadas)
fig2 = px.line(df_q2, x='Nome da Fase', y='ida_medio', color='ano',
               markers=True, text='ida_medio',
               title="2. Desempenho Acadêmico Médio (IDA) por Fase (Comparativo Anual)",
               labels={'Nome da Fase': 'Fase no Programa', 'ida_medio': 'Nota Média do IDA', 'ano': 'Ano da Avaliação'},

               # Ditando a ordem do Eixo X e da Legenda (Ano)
               category_orders={
                   "Nome da Fase": ["Alfa", "Fase 1", "Fase 2", "Fase 3", "Fase 4", "Fase 5", "Fase 6", "Fase 7", "Fase 8"],
                   "ano": ["2022", "2023", "2024"] # Legenda em ordem cronológica estrita
               },

               # Gradiente de cores (Azul Claro para o passado -> Azul Escuro/Forte para o presente)
               color_discrete_sequence=['#9ecae1', '#3182bd', '#08519c'],

               height=650, width=900)

# Ajustando a posição e tamanho do texto
fig2.update_traces(textposition='top center', textfont=dict(size=13))

# Aplicando o zoom no Eixo Y (altura)
fig2.update_layout(
    yaxis=dict(range=[min_y, max_y]),
    margin=dict(t=50)
)

fig2.show()

**Insight Analítico:** Observa-se que as Fase 6 consolida o maior Desempenho Acadêmico histórico (destaque para a média 7,23 em 2024). Nota-se também uma oscilação interessante: enquanto as Fase 2 e 3 apresentaram uma ligeira queda em 2024 face a 2023, as fases intermédias (3 a 5) mostram uma recuperação, sugerindo diferentes níveis de desafio pedagógico à medida que o aluno avança no programa.

### Pergunta 3: O grau de engajamento dos alunos (IEG) tem relação direta com seus indicadores de desempenho (IDA) e do ponto de virada (IPV)?

In [6]:
# 1. Preparando o gráfico de dispersão (Scatter Plot)
fig3 = px.scatter(df_analise,
                  x='ieg',
                  y='ida',
                  color='ipv',
                  size='ipv',
                  hover_data=['ano', 'fase'],

                  title="3. O impacto do Engajamento (IEG) no Desempenho (IDA) e Ponto de Virada (IPV)",
                  labels={
                      'ieg': 'Engajamento nas Atividades (IEG)',
                      'ida': 'Desempenho Acadêmico (IDA)',
                      'ipv': 'Ponto de Virada (IPV)'
                  },

                  # Escala de cor: Amarelo Claro = IEG Alto | Roxo/Preto Escuro = IEG Baixo
                  color_continuous_scale='Inferno',

                  height=650,
                  width=900)

fig3.update_layout(
    xaxis=dict(range=[0, 11]),
    yaxis=dict(range=[0, 11]),
    margin=dict(t=50)
)

fig3.show()

**Insight Analítico:** O gráfico de dispersão revela uma forte correlação entre os indicadores IEG, IDA e IPV. A alta concentração de 'bolhas amarelas' encontra-se no quadrante superior direito. Isso demonstra que a 'virada de chave' na vida do aluno ocorre quando há uma combinação simultânea de elevado engajamento (IEG acima de 8) e bom desempenho acadêmico (IDA acima de 7). Alunos desengajados tendem a apresentar baixos índices de IPV, representados pelos pontos mais escuros no gráfico.

### Pergunta 4: **Autoavaliação (IAA)**  As percepções dos alunos sobre si mesmos (IAA) são coerentes com seu desempenho real (IDA) e engajamento (IEG)?

In [9]:
# Gráfico de dispersão focado na Autoavaliação

fig4 = px.scatter(df_analise,
                  x='iaa',
                  y='ida',
                  color='ieg',
                  size='ieg',
                  hover_data=['ano', 'fase'],
                  opacity=0.8,

                  title="4. Coerência: A percepção do aluno (IAA) vs Realidade Acadêmica (IDA) e Engajamento (IEG)",
                  labels={
                      'iaa': 'Autoavaliação (IAA) - Como o aluno se vê',
                      'ida': 'Desempenho Acadêmico (IDA) - A realidade',
                      'ieg': 'Engajamento (IEG)'
                  },

                  # Usando a paleta 'Viridis' (do azul escuro ao amarelo claro)
                  color_continuous_scale='Viridis',

                  height=650,
                  width=900)

# Ajustando os eixos de 0 a 10 (colocamos -0.5 a 11 apenas para dar respiro nas bordas)
fig4.update_layout(
    xaxis=dict(range=[-0.5, 11]),
    yaxis=dict(range=[-0.5, 11]),
    margin=dict(t=50)
)

fig4.show()

**Insight Analítico:** Existe uma coerência geral entre o que o aluno percebe de si mesmo (IAA) e o seu desempenho real (IDA), Contudo, há casos de alunos com alta autoavaliação (IAA: 8-10) e desempenho mediano (IDA 5-6). Como esses alunos mantêm um alto nível de engajamento (cores claras no IEG), infere-se que o esforço não está a ser traduzido em notas, o que exige um acompanhamento pedagógico focado nas metodologias de estudo

### Pergunta 5: **Aspectos psicossociais (IPS)** Há padrões psicossociais (IPS) que antecedem quedas de desempenho acadêmico ou de engajamento?

In [10]:
# Ordenando por aluno e ano para buscar os dados do ano anterior
df_q5 = df_analise.sort_values(by=['ra', 'ano']).copy()
df_q5['ida_anterior'] = df_q5.groupby('ra')['ida'].shift(1)
df_q5['ieg_anterior'] = df_q5.groupby('ra')['ieg'].shift(1)

# Criando as Faixas de Nível
def nivel_ieg(nota):
    if pd.isna(nota): return 'Sem nota'
    if nota <= 5.0: return 'Crítico'
    if nota <= 7.0: return 'Alerta'
    if nota <= 8.0: return 'Bom'
    return 'Ótimo'

def nivel_ida(nota):
    if pd.isna(nota): return 'Sem nota'
    if nota <= 4.0: return 'Crítico'
    if nota <= 6.0: return 'Alerta'
    if nota <= 7.0: return 'Regular'
    if nota <= 8.0: return 'Bom'
    return 'Ótimo'

df_q5['nivel_ieg'] = df_q5['ieg'].apply(nivel_ieg)
df_q5['nivel_ida'] = df_q5['ida'].apply(nivel_ida)

# Classificando o Status Simplificado (Caiu vs Manteve/Evoluiu)
def classificar_queda_simples(row):
    if pd.isna(row['ida_anterior']) or pd.isna(row['ieg_anterior']):
        return 'Sem histórico'

    # Se a nota caiu no IDA ou no IEG
    caiu_ida = row['ida'] < row['ida_anterior']
    caiu_ieg = row['ieg'] < row['ieg_anterior']

    if caiu_ida or caiu_ieg:
        return 'Teve Queda (IDA ou IEG)'
    else:
        return 'Manteve ou Evoluiu'

df_q5['Status de Evolução'] = df_q5.apply(classificar_queda_simples, axis=1)

# Filtrar apenas 2023 e 2024 (removendo quem não tem histórico)
df_q5 = df_q5[(df_q5['ano'].astype(str).isin(['2023', '2024'])) &
              (df_q5['Status de Evolução'] != 'Sem histórico')]

# Calcular a quantidade de alunos (n) e adicionar ao rótulo do Eixo X
contagens = df_q5.groupby(['ano', 'Status de Evolução']).size().reset_index(name='qtd')
df_q5 = df_q5.merge(contagens, on=['ano', 'Status de Evolução'])
df_q5['Eixo X Dinamico'] = df_q5['Status de Evolução'] + "<br>(n=" + df_q5['qtd'].astype(str) + ")"

# ordenando a exibição (Alerta primeiro, Sucesso depois)
ordem_status = ['Teve Queda (IDA ou IEG)', 'Manteve ou Evoluiu']
df_q5['Ordem'] = df_q5['Status de Evolução'].map({k: i for i, k in enumerate(ordem_status)})
df_q5 = df_q5.sort_values(['ano', 'Ordem'])

# Criar o Boxplot Interativo
fig5 = px.box(df_q5,
              x='Eixo X Dinamico',
              y='ips',
              color='Status de Evolução',
              facet_col='ano',

              title="5. A saúde psicológica (IPS) reflete as quedas de Desempenho ou Engajamento?",
              labels={'ips': 'Nota IPS (Saúde Psicológica)', 'Eixo X Dinamico': 'Cenário do Aluno'},

              # Azul para evolução positiva, Vermelho para o Alerta de Queda
              color_discrete_map={
                  'Manteve ou Evoluiu': '#3182bd',
                  'Teve Queda (IDA ou IEG)': '#de2d26'
              },
              height=600, width=950)

# Ajustes visuais
fig5.update_layout(
    yaxis=dict(range=[0, 11]),
    yaxis2=dict(range=[0, 11]),
    showlegend=False,
    margin=dict(b=80)
)

fig5.update_xaxes(matches=None)

# Limpa os títulos superiores (Remove o "ano=")
fig5.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig5.show()

In [11]:
df_q5['nivel_ida'].value_counts()

,count
nivel_ida,
Alerta,300
Bom,218
Regular,217
Ótimo,188
Crítico,134


**Insight Analítico:** O gráfico compara os Índices Psicossociais (IPS) dos alunos que tiveram queda nas notas acadêmicas ou de engajamento (caixas vermelhas) com aqueles que mantiveram ou melhoraram suas notas (caixas azuis). A análise visual das 'caixas' demonstra que a mediana do (IPS) é estatisticamente semelhante tanto para os alunos que tiveram queda no desempenho quanto para os demais. Isso sugere que o indicador IPS, isoladamente, não possui correlação direta com a queda de Desempenho (IDA) ou Engajamento (IEG) de um ano para o outro, indicando que a raiz das quedas acadêmicas provém de outros fatores.

***Nota:*** *O número de 'Alunos Avaliados' refere-se exclusivamente aos alunos que possuíam notas registradas no ano imediatamente anterior para fins de comparação.*

### Pergunta 6: As avaliações Psicopedagógicas (IPP) confirmam ou contradizem a defasagem identificada pelo IAN?

In [12]:
# Preparando a base e recriando as categorias do IAN (para garantir que estão isoladas)
df_q6 = df_analise.copy()

def categorizar_ian(nota):
    if pd.isna(nota):
        return 'Sem nota'
    elif nota >= 9.0:
        return 'Adequado'
    elif nota >= 4.0:
        return 'Defasagem Moderada'
    else:
        return 'Defasagem Severa'

df_q6['Perfil IAN'] = df_q6['ian'].apply(categorizar_ian)

# Filtrando e removendo quem não teve nota no IAN ou IPP
df_q6 = df_q6[(df_q6['Perfil IAN'] != 'Sem nota') & (df_q6['ipp'].notna())]

contagens_q6 = df_q6.groupby('Perfil IAN').size().reset_index(name='qtd')
df_q6 = df_q6.merge(contagens_q6, on='Perfil IAN')
df_q6['Eixo X'] = df_q6['Perfil IAN'] + "<br>(n=" + df_q6['qtd'].astype(str) + ")"

# Criando o Boxplot Interativo
fig6 = px.box(df_q6,
              x='Eixo X',
              y='ipp',
              color='Perfil IAN',

              title="6. Coerência IPP vs IAN: A avaliação psicopedagógica confirma a defasagem?",
              labels={
                  'ipp': 'Nota Psicopedagógica (IPP)',
                  'Eixo X': 'Situação de Defasagem (IAN)'
              },

              # Cores: Verde (Bom), Laranja (Atenção), Vermelho (Crítico)
              color_discrete_map={
                  'Adequado': 'green',
                  'Defasagem Moderada': 'orange',
                  'Defasagem Severa': 'red'
              },

              # Forçando a ordem: Do "Melhor" para o "Pior"
              category_orders={
                  'Perfil IAN': ['Adequado', 'Defasagem Moderada', 'Defasagem Severa']
              },

              height=600, width=900)

# Ajustando o layout
fig6.update_layout(
    yaxis=dict(range=[0, 11]),
    showlegend=False
)

fig6.show()

**Insight Analítico:** O gráfico indica elevada coerência. Alunos com Nível Adequado (IAN sem defasagem) recebem notas maiores na avaliação psicopedagógica (IPP) em comparação aos alunos com defasagem 'moderada' ou 'severa'. Isso valida a metodologia de avaliação psicopedagógica aplicada pelos profissionais da Associação Passos Mágicos.

#### Pergunta 7: O que mais influencia o aluno a 'virar a chave' (IPV)?

In [14]:
# ==========================================
# GRÁFICO 7.1: CORRELAÇÃO DOS FATORES NO IPV
# ==========================================

# Selecionando as colunas de comportamento (Acadêmico, Engajamento, Psico)
colunas_comportamento = ['ida', 'ieg', 'ips', 'ipp', 'iaa']
df_corr = df_analise[colunas_comportamento + ['ipv']].dropna()

# Calculando a correlação de todas as variáveis com o IPV
correlacoes = df_corr.corr()['ipv'].drop('ipv').sort_values(ascending=True).reset_index()
correlacoes.columns = ['Comportamento', 'Força da Influência']

# Renomeando os nomes das colunas para a visualização
nomes_amigaveis = {
    'ida': 'Desempenho Acadêmico (IDA)',
    'ieg': 'Engajamento (IEG)',
    'ips': 'Saúde Psicossocial (IPS)',
    'ipp': 'Avaliação Psicopedagógica (IPP)',
    'iaa': 'Autoavaliação (IAA)'
}
correlacoes['Comportamento'] = correlacoes['Comportamento'].map(nomes_amigaveis)

# Criando o Gráfico de Barras Horizontais
fig7 = px.bar(correlacoes,
              x='Força da Influência',
              y='Comportamento',
              orientation='h',
              title="7. Fatores que mais impulsionam o Ponto de Virada (IPV)",
              text_auto='.2f',
              color='Força da Influência',
              color_continuous_scale='Blues',
              height=600, width=1000)

# Ajustes estéticos
fig7.update_layout(
    xaxis_title="Grau de Influência (Correção de Pearson)",
    yaxis_title="",
    coloraxis_showscale=False
)

fig7.show()

**Insight Analítico (Visão Geral):** Numa perspetiva histórica dos anos (2022 a 2024) somados, a Avaliação Pscicopedagógica (IPP), o Desempenho Acadêmico (IDA) e o Engajamento (IEG) aparecem como os fatores mais fortemente correlacionados com o Ponto de Virada (IPV). Isto indica que a perfomance e a participação ativa dos alunos são a base para a transformação de mentalidade, o que tem sido capturado pela equipe psicopedagógica. Por outro lado, os resultados de autoavaliação e os fatores psicossociais dos alunos apontam não haver correlação direta com o IPV.

In [15]:
# ==========================================
# GRÁFICO 7.2: INFLUÊNCIA (PESO) DOS FATORES NO IPV AO LONGO DO TEMPO
# ==========================================
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import numpy as np

# Preparação dos dados
indicadores = ['ida', 'ieg', 'ips', 'ipp', 'iaa']
anos = [2022, 2023, 2024]
dados_influencia = []

for ano in anos:
    # Filtra o ano e remove nulos para o modelo
    df_ano = df_analise[df_analise['ano'] == ano].dropna(subset=indicadores + ['ipv'])

    if len(df_ano) > 10:
        # Identifica quais indicadores variam (evita erro com o IPS constante de 2022)
        indicadores_validos = [ind for ind in indicadores if df_ano[ind].std() > 0]

        X = df_ano[indicadores_validos]
        y = df_ano['ipv']

        # PADRONIZAÇÃO:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        y_scaled = (y - y.mean()) / y.std()

        # Treina uma Regressão Linear para extrair os pesos (influência)
        modelo = LinearRegression()
        modelo.fit(X_scaled, y_scaled)

        # Salva os coeficientes
        for i, ind in enumerate(indicadores_validos):
            if ano == 2022 and ind in ['ipp']:
                continue
            dados_influencia.append({
                'Ano': str(ano),
                'Indicador': ind,
                'Grau de Influência': round(modelo.coef_[i], 3)
            })

# DataFrame para o Gráfico
df_influencia = pd.DataFrame(dados_influencia)
nomes_map = {
    'ida': 'Desempenho (IDA)', 'ieg': 'Engajamento (IEG)',
    'ips': 'Psicossocial (IPS)', 'ipp': 'Psicopedagógico (IPP)',
    'iaa': 'Autoavaliação (IAA)'
}
df_influencia['Indicador'] = df_influencia['Indicador'].map(nomes_map)

# Gráfico de Evolução da Influência
fig7_influencia = px.line(df_influencia,
                          x='Ano',
                          y='Grau de Influência',
                          color='Indicador',
                          markers=True,
                          text='Grau de Influência',
                          title="7. Evolução da Influência Real dos Comportamentos no Ponto de Virada (IPV)",
                          labels={'Grau de Influência': 'Peso da Influência (Coeficientes Beta)'},
                          height=650, width=950)

# O range de -0.2 a 1.2
fig7_influencia.update_layout(
    yaxis=dict(range=[-0.2, 1.2], zeroline=True, zerolinewidth=2, zerolinecolor='white'),
    margin=dict(t=50, b=50),
    legend_title_text='Indicadores'
)

# Ajuste da posição do texto
fig7_influencia.update_traces(textposition='top center', textfont=dict(size=11))

fig7_influencia.show()

**Insight Analítico (Visão Evolutiva):** A quebra temporal através da regressão revela que o Desempenho acadêmico (IDA) e o Engajamento (IEG), este mais fortemente a partir de 2023, são os fatores preponderantes de correlação com o ponto de virada (IPV). Essa correlação passou a ser melhor capturada pela avaliação psicopedagógica (IPP), que pasou a ser realizada a partir de 2023.

#### Pergunta 8: Quais combinações de indicadores (IDA + IEG + IPS + IPP) elevam mais a nota global do aluno (INDE)?

In [16]:
#Filtro danos: 2023 e 2024
df_q8 = df_analise[df_analise['ano'].astype(str).isin(['2023', '2024'])].copy()

# Definindo a "Alta Performance" (Nota >= 8)
c_ida = df_q8['ida'] >= 8
c_ieg = df_q8['ieg'] >= 8
c_ips = df_q8['ips'] >= 8
c_ipp = df_q8['ipp'] >= 8

# Calculando a média do INDE para TODAS as combinações relevantes
combos = [
    {'Nome': 'Super Combo (Os 4 indicadores >= 8)', 'Media': df_q8[c_ida & c_ieg & c_ips & c_ipp]['inde'].mean()},
    {'Nome': 'IDA + IPS (Acadêmico + Psicossocial)', 'Media': df_q8[c_ida & c_ips]['inde'].mean()},
    {'Nome': 'IEG + IPS (Engajamento + Psicossocial)', 'Media': df_q8[c_ieg & c_ips]['inde'].mean()},
    {'Nome': 'IDA + IPP (Acadêmico + Psicopedagógico)', 'Media': df_q8[c_ida & c_ipp]['inde'].mean()},
    {'Nome': 'IPS + IPP (Psicossocial + Psicopedagógico)', 'Media': df_q8[c_ips & c_ipp]['inde'].mean()},
    {'Nome': 'IDA + IEG (Acadêmico + Engajamento)', 'Media': df_q8[c_ida & c_ieg]['inde'].mean()},
    {'Nome': 'IEG + IPP (Engajamento + Psicopedagógico)', 'Media': df_q8[c_ieg & c_ipp]['inde'].mean()}
]

# Criando o DataFrame e ordenando pela nota (do menor para o maior para o gráfico subir)
df_combos = pd.DataFrame(combos).dropna().sort_values(by='Media', ascending=True)

# Gráfico de Barras Horizontais
fig8 = px.bar(df_combos,
              x='Media',
              y='Nome',
              orientation='h',
              title="8. Quais combinações de indicadores mais elevam o INDE?",
              text_auto='.2f',
              color='Media',
              color_continuous_scale='RdPu',
              labels={'Media': 'Nota Média do INDE', 'Nome': 'Combinações (Notas >= 8)'})

# Ajustes de layout
fig8.update_layout(
    xaxis=dict(range=[0, 10.5], title="Nota Global Média Resultante (INDE)"),
    yaxis=dict(title="Combinação de Alta Performance"),
    coloraxis_showscale=False,
    margin=dict(l=50, r=50, t=80, b=50),
    height=600
)

fig8.show()

**Insight Analítico:** Como esperado o 'Super Combo' (quatro fatores, com valores acima de 8) resulta no maior Índice Global de Desenvolvimento Educacional (INDE). No entanto, a grande descoberta é a combinação do IDA + IPS (Desempenho Académico + Avaliação Psicossocial), que isoladamente apresentou um INDE médio (8.71), muito próximo ao Super Combo (8.93), provando que um ótimo desempenho acadêmico aliado a uma base psicossocial sólida possibilita ao aluno um resultado global de excelência no programa, mesmo que o engajamento ou o IPP oscilem.